In [ ]:
from __future__ import annotations

import os
import hashlib
from pathlib import Path
from typing import Iterable

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# ============================================================
# User settings
# ============================================================

def resolve_repository_root() -> Path:
    root = Path(
        os.environ.get(
            "PHYLOGENY_REPOSITORY_ROOT",
            Path.cwd(),
        )
    ).expanduser().resolve()

    if root.name in {"AA", "NT", "common"}:
        root = root.parent

    return root


REPOSITORY_ROOT = resolve_repository_root()

PROJECT_ROOTS = {
    "AA": REPOSITORY_ROOT / "AA",
    "NT": REPOSITORY_ROOT / "NT",
}

# Set a Path explicitly to avoid accidentally reading an older evaluation.
# None means: select the newest CSV containing both method == SQBM and NJ.
EVALUATION_CSV = {
    "AA": None,
    "NT": None,
}

OUTPUT_ROOT = REPOSITORY_ROOT / "paper_analysis_nj_rep100"

N_BOOT = 10_000
BASE_SEED = 20260731
EXPECTED_REPS_PER_CONDITION = 100

COMPARISONS = {
    "AA": (
        {
            "comparison": "Log-local",
            "qubo_variant": "logkernel_nmcut",
            "nj_variant": "log_nj",
        },
        {
            "comparison": "Log-global",
            "qubo_variant": "logglobal_nmcut",
            "nj_variant": "log_nj",
        },
        {
            "comparison": "ER20-local",
            "qubo_variant": "poisson20_selftune_nmcut",
            "nj_variant": "er20_nj",
        },
        {
            "comparison": "WAG-local",
            "qubo_variant": "wag_selftune_nmcut",
            "nj_variant": "wag_nj",
        },
    ),
    "NT": (
        {
            "comparison": "Log-local",
            "qubo_variant": "logkernel_nmcut",
            "nj_variant": "log_nj",
        },
        {
            "comparison": "Log-global",
            "qubo_variant": "logglobal_nmcut",
            "nj_variant": "log_nj",
        },
        {
            "comparison": "JC69-local",
            "qubo_variant": "jc69_selftune_nmcut",
            "nj_variant": "jc69_nj",
        },
    ),
}

GENERATORS = ("rtree", "yule")
BRANCH_LENGTHS = (0.125, 0.250, 0.500, 0.625, 0.750)


In [ ]:
# ============================================================
# Utilities
# ============================================================

def stable_seed(*parts: object) -> int:
    text = "|".join(str(x) for x in (BASE_SEED, *parts))
    digest = hashlib.sha256(text.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], "little") % (2**32)


def discover_combined_evaluation(project_root: Path) -> Path:
    candidates = sorted(
        (project_root / "results_evaluation").glob("evaluation_*.csv"),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )

    for path in candidates:
        try:
            frame = pd.read_csv(path, usecols=["method", "status"])
        except Exception:
            continue

        methods = set(frame.loc[frame["status"] == "ok", "method"].astype(str))
        if {"SQBM", "NJ"}.issubset(methods):
            return path

    raise FileNotFoundError(
        f"No combined SQBM/NJ evaluation CSV found under "
        f"{project_root / 'results_evaluation'}"
    )


def load_evaluation(sequence_type: str) -> tuple[pd.DataFrame, Path]:
    path = EVALUATION_CSV[sequence_type]
    if path is None:
        path = discover_combined_evaluation(PROJECT_ROOTS[sequence_type])
    path = Path(path)

    frame = pd.read_csv(path)
    required = {
        "sequence_type",
        "method",
        "tag",
        "generator",
        "bl",
        "rep",
        "variant",
        "postswap_mode",
        "status",
        "recovered_branch_percent",
    }
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {sorted(missing)}")

    frame = frame.loc[frame["status"] == "ok"].copy()
    frame["sequence_type"] = sequence_type
    frame["bl"] = pd.to_numeric(frame["bl"], errors="raise")
    frame["rep"] = pd.to_numeric(frame["rep"], errors="raise").astype(int)
    frame["recovered_branch_percent"] = pd.to_numeric(
        frame["recovered_branch_percent"], errors="raise"
    )
    return frame, path


def bootstrap_mean_ci(values: np.ndarray, *, seed: int) -> tuple[float, float]:
    values = np.asarray(values, dtype=float)
    if values.size == 0:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    boot = np.empty(N_BOOT, dtype=float)
    n = values.size

    for i in range(N_BOOT):
        boot[i] = rng.choice(values, size=n, replace=True).mean()

    return tuple(np.quantile(boot, [0.025, 0.975]))


def stratified_bootstrap_ci(
    frame: pd.DataFrame,
    value_col: str,
    *,
    seed: int,
) -> tuple[float, float]:
    strata = [
        group[value_col].to_numpy(dtype=float)
        for _, group in frame.groupby(["generator", "bl"], sort=True)
    ]
    if not strata:
        return np.nan, np.nan

    rng = np.random.default_rng(seed)
    boot = np.empty(N_BOOT, dtype=float)

    for i in range(N_BOOT):
        stratum_means = [
            rng.choice(values, size=len(values), replace=True).mean()
            for values in strata
        ]
        boot[i] = np.mean(stratum_means)

    return tuple(np.quantile(boot, [0.025, 0.975]))


In [ ]:
# ============================================================
# Pair construction
# ============================================================

def build_pairs(frame: pd.DataFrame, sequence_type: str) -> pd.DataFrame:
    outputs = []
    keys = ["sequence_type", "tag", "generator", "bl", "rep"]

    for spec in COMPARISONS[sequence_type]:
        qubo = frame.loc[
            (frame["method"] == "SQBM")
            & (frame["postswap_mode"] == "none")
            & (frame["variant"] == spec["qubo_variant"]),
            keys + ["recovered_branch_percent"],
        ].rename(
            columns={"recovered_branch_percent": "qubo_recovery_percent"}
        )

        nj = frame.loc[
            (frame["method"] == "NJ")
            & (frame["variant"] == spec["nj_variant"]),
            keys + ["recovered_branch_percent"],
        ].rename(
            columns={"recovered_branch_percent": "nj_recovery_percent"}
        )

        if qubo.duplicated(keys).any():
            raise ValueError(f"Duplicate SQBM rows: {sequence_type} {spec['comparison']}")
        if nj.duplicated(keys).any():
            raise ValueError(f"Duplicate NJ rows: {sequence_type} {spec['comparison']}")

        paired = qubo.merge(nj, on=keys, how="inner", validate="one_to_one")
        paired["comparison"] = spec["comparison"]
        paired["qubo_variant"] = spec["qubo_variant"]
        paired["nj_variant"] = spec["nj_variant"]
        paired["difference_pp"] = (
            paired["qubo_recovery_percent"] - paired["nj_recovery_percent"]
        )
        outputs.append(paired)

    return pd.concat(outputs, ignore_index=True)


def make_integrity_table(pairs: pd.DataFrame) -> pd.DataFrame:
    result = (
        pairs.groupby(
            ["sequence_type", "comparison", "generator", "bl"],
            as_index=False,
        )
        .agg(
            n_pairs=("tag", "size"),
            n_unique_tags=("tag", "nunique"),
        )
    )
    result["expected_n"] = EXPECTED_REPS_PER_CONDITION
    result["complete"] = (
        (result["n_pairs"] == EXPECTED_REPS_PER_CONDITION)
        & (result["n_unique_tags"] == EXPECTED_REPS_PER_CONDITION)
    )
    return result


In [ ]:
# ============================================================
# Summaries
# ============================================================

def summarize_conditions(pairs: pd.DataFrame) -> pd.DataFrame:
    rows = []
    group_cols = ["sequence_type", "comparison", "generator", "bl"]

    for key, group in pairs.groupby(group_cols, sort=True):
        sequence_type, comparison, generator, bl = key
        diff = group["difference_pp"].to_numpy(dtype=float)
        ci_low, ci_high = bootstrap_mean_ci(
            diff,
            seed=stable_seed("condition", *key),
        )
        tolerance = 1e-12

        rows.append(
            {
                "sequence_type": sequence_type,
                "comparison": comparison,
                "generator": generator,
                "bl": bl,
                "n_pairs": len(group),
                "qubo_mean": group["qubo_recovery_percent"].mean(),
                "nj_mean": group["nj_recovery_percent"].mean(),
                "mean_difference_pp": diff.mean(),
                "sd_difference_pp": diff.std(ddof=1),
                "median_difference_pp": np.median(diff),
                "ci_low_pp": ci_low,
                "ci_high_pp": ci_high,
                "qubo_better": int(np.sum(diff > tolerance)),
                "equal": int(np.sum(np.abs(diff) <= tolerance)),
                "nj_better": int(np.sum(diff < -tolerance)),
            }
        )

    return pd.DataFrame(rows).sort_values(group_cols).reset_index(drop=True)


def summarize_overall(pairs: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for key, group in pairs.groupby(
        ["sequence_type", "comparison"], sort=True
    ):
        sequence_type, comparison = key
        ci_low, ci_high = stratified_bootstrap_ci(
            group,
            "difference_pp",
            seed=stable_seed("overall", *key),
        )

        # Equal weight for the ten generator × branch-length strata.
        stratum_summary = (
            group.groupby(["generator", "bl"], as_index=False)
            .agg(
                qubo_mean=("qubo_recovery_percent", "mean"),
                nj_mean=("nj_recovery_percent", "mean"),
                difference_mean=("difference_pp", "mean"),
            )
        )
        diff = group["difference_pp"].to_numpy(dtype=float)
        tolerance = 1e-12

        rows.append(
            {
                "sequence_type": sequence_type,
                "comparison": comparison,
                "n_pairs": len(group),
                "n_strata": len(stratum_summary),
                "qubo_mean_equal_strata": stratum_summary["qubo_mean"].mean(),
                "nj_mean_equal_strata": stratum_summary["nj_mean"].mean(),
                "mean_difference_pp_equal_strata": stratum_summary[
                    "difference_mean"
                ].mean(),
                "ci_low_pp": ci_low,
                "ci_high_pp": ci_high,
                "qubo_better": int(np.sum(diff > tolerance)),
                "equal": int(np.sum(np.abs(diff) <= tolerance)),
                "nj_better": int(np.sum(diff < -tolerance)),
            }
        )

    return pd.DataFrame(rows)


def summarize_absolute_conditions(pairs: pd.DataFrame) -> pd.DataFrame:
    return (
        pairs.groupby(
            ["sequence_type", "comparison", "generator", "bl"],
            as_index=False,
        )
        .agg(
            n_pairs=("tag", "size"),
            qubo_mean=("qubo_recovery_percent", "mean"),
            qubo_sd=("qubo_recovery_percent", "std"),
            nj_mean=("nj_recovery_percent", "mean"),
            nj_sd=("nj_recovery_percent", "std"),
        )
    )


In [ ]:
# ============================================================
# Paper figures
# ============================================================

def save_figure(fig: plt.Figure, path_stem: Path) -> None:
    path_stem.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path_stem.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(path_stem.with_suffix(".png"), dpi=300, bbox_inches="tight")
    plt.close(fig)


def plot_difference(
    condition_summary: pd.DataFrame,
    *,
    sequence_type: str,
    generator: str,
    output_dir: Path,
) -> None:
    data = condition_summary.loc[
        (condition_summary["sequence_type"] == sequence_type)
        & (condition_summary["generator"] == generator)
    ].copy()

    fig, ax = plt.subplots(figsize=(7.0, 4.8))

    for comparison, group in data.groupby("comparison", sort=False):
        group = group.sort_values("bl")
        yerr = np.vstack(
            [
                group["mean_difference_pp"] - group["ci_low_pp"],
                group["ci_high_pp"] - group["mean_difference_pp"],
            ]
        )
        ax.errorbar(
            group["bl"],
            group["mean_difference_pp"],
            yerr=yerr,
            marker="o",
            capsize=3,
            label=comparison,
        )

    ax.axhline(0.0, linewidth=1.0, linestyle="--")
    ax.set_xlabel("Mean branch length")
    ax.set_ylabel("Split recovery difference (SQBM − NJ, percentage points)")
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()

    save_figure(
        fig,
        output_dir / f"fig_nj_difference_{sequence_type.lower()}_{generator}",
    )


def plot_absolute(
    absolute_summary: pd.DataFrame,
    *,
    sequence_type: str,
    generator: str,
    comparison: str,
    output_dir: Path,
) -> None:
    data = absolute_summary.loc[
        (absolute_summary["sequence_type"] == sequence_type)
        & (absolute_summary["generator"] == generator)
        & (absolute_summary["comparison"] == comparison)
    ].sort_values("bl")

    fig, ax = plt.subplots(figsize=(6.4, 4.6))
    ax.plot(data["bl"], data["qubo_mean"], marker="o", label="SQBM recursive")
    ax.plot(data["bl"], data["nj_mean"], marker="s", label="NJ")
    ax.set_xlabel("Mean branch length")
    ax.set_ylabel("Split recovery (%)")
    ax.set_ylim(0, 100)
    ax.legend(frameon=False)
    ax.grid(axis="y", alpha=0.25)
    fig.tight_layout()

    safe_comparison = comparison.lower().replace("-", "_").replace(" ", "_")
    save_figure(
        fig,
        output_dir
        / f"fig_absolute_{sequence_type.lower()}_{generator}_{safe_comparison}",
    )


def plot_overall_forest(overall: pd.DataFrame, output_dir: Path) -> None:
    data = overall.copy()
    data["label"] = data["sequence_type"] + ": " + data["comparison"]
    data = data.sort_values(
        ["sequence_type", "mean_difference_pp_equal_strata"]
    ).reset_index(drop=True)

    y = np.arange(len(data))
    mean = data["mean_difference_pp_equal_strata"].to_numpy()
    xerr = np.vstack(
        [
            mean - data["ci_low_pp"].to_numpy(),
            data["ci_high_pp"].to_numpy() - mean,
        ]
    )

    fig, ax = plt.subplots(figsize=(7.2, max(4.2, 0.55 * len(data) + 1.5)))
    ax.errorbar(mean, y, xerr=xerr, fmt="o", capsize=3)
    ax.axvline(0.0, linewidth=1.0, linestyle="--")
    ax.set_yticks(y, data["label"])
    ax.set_xlabel("Mean split recovery difference (SQBM − NJ, percentage points)")
    ax.grid(axis="x", alpha=0.25)
    fig.tight_layout()

    save_figure(fig, output_dir / "fig_nj_overall_forest")


In [ ]:
# ============================================================
# Main
# ============================================================

def main() -> None:
    table_dir = OUTPUT_ROOT / "tables"
    figure_dir = OUTPUT_ROOT / "figures"
    table_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)

    paired_frames = []
    input_records = []

    for sequence_type in ("AA", "NT"):
        evaluation, path = load_evaluation(sequence_type)
        input_records.append(
            {"sequence_type": sequence_type, "evaluation_csv": str(path)}
        )
        paired_frames.append(build_pairs(evaluation, sequence_type))

    pairs = pd.concat(paired_frames, ignore_index=True)
    integrity = make_integrity_table(pairs)

    if not integrity["complete"].all():
        incomplete = integrity.loc[~integrity["complete"]]
        incomplete.to_csv(table_dir / "incomplete_pairing_conditions.csv", index=False)
        raise RuntimeError(
            "Incomplete Reps pairing. See incomplete_pairing_conditions.csv"
        )

    condition_summary = summarize_conditions(pairs)
    overall_summary = summarize_overall(pairs)
    absolute_summary = summarize_absolute_conditions(pairs)

    pd.DataFrame(input_records).to_csv(
        table_dir / "analysis_inputs.csv", index=False
    )
    pairs.to_csv(table_dir / "nj_pairwise_tree_level_rep100.csv", index=False)
    integrity.to_csv(table_dir / "nj_pairing_integrity_rep100.csv", index=False)
    condition_summary.to_csv(
        table_dir / "nj_condition_paired_summary_rep100.csv", index=False
    )
    overall_summary.to_csv(
        table_dir / "nj_overall_paired_summary_rep100.csv", index=False
    )
    absolute_summary.to_csv(
        table_dir / "nj_absolute_condition_summary_rep100.csv", index=False
    )

    for sequence_type in ("AA", "NT"):
        for generator in GENERATORS:
            plot_difference(
                condition_summary,
                sequence_type=sequence_type,
                generator=generator,
                output_dir=figure_dir,
            )

        for spec in COMPARISONS[sequence_type]:
            for generator in GENERATORS:
                plot_absolute(
                    absolute_summary,
                    sequence_type=sequence_type,
                    generator=generator,
                    comparison=spec["comparison"],
                    output_dir=figure_dir,
                )

    plot_overall_forest(overall_summary, figure_dir)

    print("\n== Input evaluation files ==")
    print(pd.DataFrame(input_records).to_string(index=False))
    print("\n== Overall paired summary ==")
    print(overall_summary.to_string(index=False))
    print(f"\n[saved] {OUTPUT_ROOT}")


if __name__ == "__main__":
    main()
